# CS 495 Capstone in Data Science
## Project Proposal & Planning Document

**Binomial Option Pricing in Practice: Using CRR American Option Valuation to Detect Mispricing and Inform a Profit-Maximizing Trade Strategy on AMD Options**

Bellevue College | Spring 2026

---

## 1. What Is Your Project?

This capstone implements the Cox-Ross-Rubinstein (CRR) Binomial Option Pricing Model for American-style equity options and applies the resulting theoretical prices as a probability estimator inside a Kelly Criterion trade-sizing framework — mirroring the methodology of Project Suggestion 6 (*Profit-Maximizing Model for Prediction Markets*).

> **Core question:** When the AMD options market prices a contract differently from what the CRR binomial model says it is worth, is that difference exploitable — and if so, how large a position does the Kelly Criterion recommend?

The project targets a live AMD option chain (stock price ~$275.10, May 8 2026 expiry) and focuses on four specific trade tickets:

| Ticket | Action | Strike | Type | Limit | Break-Even |
|--------|--------|--------|------|-------|------------|
| 3 | Sell to Open | $277.50 | Call | $14.25 | $291.75 |
| 4 | Buy to Open | $280.00 | Put | $18.55 | $261.45 |
| 5 | Buy to Open | $280.00 | Call | $14.20 | $294.20 |
| 6 | Sell to Open | $280.00 | Put | $17.87 | $262.13 |

**Scope restrictions (explicit):**
- **American options only.** All four AMD contracts are treated as American-style contracts. Early exercise capability is required in every model variant. European-style pricing is excluded.
- **No dividends.** AMD's dividend treatment is excluded from all model runs. The standard CRR risk-neutral probability formula is used without a continuous yield adjustment: `p = (exp(r·Δt) − d) / (u − d)`.

---

## 2. Motivation: Why Is This Problem Important?

### Options Markets Are Prediction Markets

The premium paid for an AMD call or put encodes the crowd's collective belief about where the stock will be at expiration. Like prediction markets, options prices are shaped by behavioral forces that create systematic mispricings:

- **Implied volatility overpricing:** AMD's chain shows IV ≈ 64–65%, which historically exceeds realized volatility during non-earnings periods. The crowd is overpricing uncertainty.
- **Strike clustering:** Volume and open interest concentrate around round strikes (e.g., $280), creating liquidity-driven distortions that a fundamental model does not predict.
- **Favorite-longshot bias:** Short-dated out-of-the-money puts are frequently overpriced relative to their true probability of finishing in-the-money.

### American Options Require the Binomial Model

Black-Scholes cannot price American options correctly because it has no mechanism for early exercise. The CRR binomial model handles early exercise natively through backward induction — at every node, the model compares the continuation value against the intrinsic value and takes the maximum. This matters practically: the AMD put options in this project can be exercised early if AMD's stock falls far enough below $280.

### Disciplined Sizing Matters When Stakes Are High

The AMD trade tickets show Max Loss values ranging from −$4,260 (capped, long call) to −$78,639 (short put). A theoretical price advantage is worthless without a framework for converting it into a position size that respects risk. The Kelly Criterion provides that framework rigorously — maximizing the long-run growth rate of wealth while preventing the catastrophic losses that come from overbetting on uncertain edges.

---

## 3. Clear Goals and Expected Impact

### Primary Goals

- **Pricing:** Implement the CRR American binomial tree in Python and validate it against observed AMD premiums and Greeks within ±5%.
- **Mispricing detection:** Compute the signed edge `(V_model − V_market) / V_market` for all four trade tickets and determine the direction of crowd bias.
- **Trade sizing:** Apply the Kelly Criterion (full, half, and quarter-Kelly) to translate each edge into a concrete dollar position recommendation.
- **Early exercise:** Map the critical stock price S\* below which early exercise of the AMD American puts is optimal, at every time step.
- **Sensitivity:** Quantify how stable the edge and Kelly fraction are under perturbations to σ, r, and T.

### Expected Impact

- Demonstrates a rigorous, end-to-end pipeline from raw market data to a disciplined trade decision.
- Bridges the gap between academic option pricing theory and what practitioners see on trading platforms like Fidelity.
- Operationalizes the Project Suggestion 6 prediction-market methodology in a live equity options context.
- Produces a reproducible Python codebase that can be adapted to other tickers, strikes, and expiries.

---

## 4. Examples and Use Cases

- **Use Case 1 — Sell Call $277.50 (Ticket 3):** Model prices the call at $13.80 vs. the observed $14.25 limit. Edge = +3.2% (market is overpricing). Kelly recommends a fractional sell position. The model identifies that IV overpricing is the primary driver of the gap.
- **Use Case 2 — Buy Put $280 (Ticket 4):** Model prices the put at $19.10 vs. the observed $18.55. Edge = −2.9% (market is underpricing). Kelly recommends no position or a minimal one. The early exercise boundary is plotted, showing that exercise becomes optimal below $263.
- **Use Case 3 — Sensitivity stress test:** σ is increased by 10%. The model price rises, the edge compresses, and the Kelly fraction shrinks — demonstrating that the sell-call recommendation depends critically on the IV assumption.
- **Use Case 4 — Convergence validation:** The American binomial price is computed for N = 10, 25, 50, 100, 200 steps to confirm numerical stability before using N=100 as the production setting.

---

## 5. Potential Datasets

### Primary Data — AMD Fidelity Option Chain (Live, April 2026)

- AMD stock price: $275.10
- Strikes: $262.50 – $287.50, May 8 2026 expiry (American style)
- Market premiums (limit prices), IVs (~64–65%), open interest, volume
- Greeks reported: Δ, Γ, θ, ν, ρ per strike
- Put/Call ratio: 0.55
- Source: Fidelity trade ticket interface (screenshot / manual extraction)
- Size: < 100 rows; computation is primarily model-generated

### Supplementary Inputs

- Risk-free rate: U.S. 3-month T-bill yield from FRED (~5.3% at data collection)
- AMD 30-day and 60-day historical realized volatility via yfinance
- *No dividend data is used in any model run*

### Limitations

- IV is a point-in-time snapshot; market conditions may shift before expiry.
- Observed premiums reflect bid-ask spreads and liquidity premiums not captured by BOPM.
- The option chain covers only one expiry cycle; results are not generalizable without recalibration.

---

## 6. Data Preprocessing Steps

- Extract AMD option chain fields from Fidelity deck: strike, expiry, bid, ask, IV, Greeks.
- Compute limit price midpoint = (bid + ask) / 2 for each contract to reduce bid-ask noise.
- Calculate time to expiration T = (May 8 2026 − collection date) / 365.
- Source risk-free rate r from FRED 3-month T-bill; confirm rate applies to the expiry window.
- Download AMD historical closing prices via yfinance; compute 30-day and 60-day realized volatility as annualized rolling standard deviation of log returns.
- Store all inputs in a structured pandas DataFrame; document the exact timestamp of data collection for reproducibility.
- Validate that put-call parity approximately holds across the chain as a sanity check on input quality.

---

## 7. Proposed Methods and Models

### CRR American Binomial Tree — Core Pricer

The Cox-Ross-Rubinstein binomial model was selected because it natively handles American-style early exercise through backward induction — a capability Black-Scholes lacks. At each of N time steps, the stock price either moves up by `u = exp(σ√Δt)` or down by `d = 1/u`. The risk-neutral probability of an up-move is:

```
p = (exp(r·Δt) − d) / (u − d)      [no dividend adjustment]
```

At each node during backward induction, the American option value is:

```
V = max(intrinsic value,  exp(−r·Δt) · (p·V_up + (1−p)·V_down))
```

### Greeks — Numerical Finite Differences

All five Greeks are computed numerically by perturbing each input variable:

| Greek | Formula |
|-------|---------|
| Delta | `Δ = (V(S+δ) − V(S−δ)) / (2δ)` |
| Gamma | `Γ = (V(S+δ) − 2V(S) + V(S−δ)) / δ²` |
| Theta | `θ = (V(T−δt) − V(T)) / δt` |
| Vega | `ν = (V(σ+0.01) − V(σ−0.01)) / 0.02` |
| Rho | `ρ = (V(r+0.01) − V(r−0.01)) / 0.02` |

### Mispricing Edge Computation

```
edge = (V_model − V_market) / V_market
```

A positive edge indicates the contract is cheap relative to the model (favors buying); a negative edge indicates it is expensive (favors selling). The edge is the bridge between the pricing model and the Kelly framework.

### Kelly Criterion — Trade Sizing

The edge is converted into a win probability p using the payoff structure of the trade. Full, half, and quarter-Kelly fractions are then computed:

```
f*        = (p·b − q) / b      [full Kelly]
f_half    = f* / 2             [half-Kelly]
f_quarter = f* / 4             [quarter-Kelly]
```

Fractional Kelly variants are evaluated because full Kelly is highly sensitive to errors in the probability estimate.

### Monte Carlo P&L Simulation

1,000 hypothetical repeat trades are simulated under each Kelly variant to produce a P&L distribution, enabling evaluation of Sharpe-like ratio, maximum drawdown, and hit rate.

---

## 8. Features Planned and Engineered

### Raw Inputs (from Option Chain)

- **S** — current AMD stock price ($275.10)
- **K** — strike price ($277.50 or $280.00)
- **σ** — implied volatility per strike (~0.64–0.65)
- **r** — risk-free rate (~0.053)
- **T** — time to expiration in years
- **V_market** — observed limit price (premium)

### Engineered Features

- **Moneyness:** `(S − K) / K` — how far in or out of the money the contract is
- **IV Premium:** AMD IV minus AMD 30-day realized volatility — the crowd bias signal
- **Edge:** `(V_model − V_market) / V_market` — the mispricing signal per ticket
- **Kelly fraction f\*:** derived from edge and payoff structure
- **Early exercise boundary S\*:** at each time step, for put options
- **Delta-neutral threshold:** stock price at which Δ = 0.50 (at-the-money crossing)

---

## 9. Evaluation Metrics

| Metric Category | Measure / Formula |
|-----------------|-------------------|
| Pricing accuracy | RMSE / MAE, Percentage Error: `|V_model − V_market| / V_market × 100` |
| Early exercise | Critical stock price S\* at each time step (American put boundary) |
| Greeks validation | Signed error vs. AMD deck: Δ, Γ, θ, ν, ρ |
| Mispricing edge | `(V_model − V_market) / V_market` — signed, per ticket |
| IV premium | AMD IV minus AMD 30-day realized volatility |
| Kelly sizing | f\*, f_half, f_quarter; implied dollar position ($100K base) |
| P&L distribution | Monte Carlo Sharpe-like ratio, max drawdown, hit rate |
| Sensitivity | Variance of f\* across σ ±10%, r ±1%, T ±5 days |

The primary success criterion is that the binomial model produces theoretical prices within ±5% of observed limit prices for all four AMD trade tickets, and that the Kelly framework produces non-trivial and stable position recommendations.

---

## 10. Project Structure and Workflow

The project follows a linear pipeline of six stages, each building directly on the previous:

1. **Data ingestion** — Extract AMD chain inputs; source r and realized volatility.
2. **Tree construction** — Build CRR lattice (NumPy vectorized) for each ticket.
3. **Pricing & Greeks** — Backward induction with American exercise; numerical Greeks.
4. **Edge computation** — Compare V_model vs. V_market; compute mispricing signal.
5. **Kelly sizing** — Convert edge to f\*, f_half, f_quarter; dollar position recommendation.
6. **Validation & sensitivity** — Monte Carlo P&L simulation; σ, r, T perturbation analysis.

The Python repository is organized with one module per stage:

```
project/
├── data.py          # Data ingestion and preprocessing
├── tree.py          # CRR binomial tree construction
├── greeks.py        # Numerical Greeks via finite differences
├── edge.py          # Mispricing edge computation
├── kelly.py         # Kelly Criterion sizing framework
├── simulation.py    # Monte Carlo P&L simulation
├── main.py          # End-to-end pipeline runner
└── config.yaml      # All inputs: S, K, σ, r, T, N
```

---

## 11. Tools, Libraries, and Technologies

### Core Stack

| Tool / Library | Purpose |
|----------------|---------|
| Python 3.11 | Primary language |
| numpy | Vectorized binomial lattice construction; numerical finite differences |
| scipy | Statistical functions; Monte Carlo helper distributions |
| pandas | Option chain data management; tabular results |
| matplotlib / plotly | Convergence plots, lattice visualization, Greek surfaces, Kelly charts, tornado charts |
| yfinance | AMD historical price data and realized volatility download |

### Optional / Supplementary

| Tool | Purpose |
|------|---------|
| streamlit | Interactive dashboard: input AMD strike/expiry → model price → edge → Kelly recommendation |
| jupyter | Notebook-based reproducible analysis for reporting |
| GitHub | Repository hosting and version control |

### Infrastructure

CPU-only; no GPU or cloud compute required. All binomial trees at N=200 complete in under 1 second with NumPy vectorization. Monte Carlo with 1,000 paths runs in under 5 seconds on a standard laptop.

---

## 12. Expected Deliverables

### Required

- Research report (10–20 pages) covering theory, implementation, results, pricing validation, mispricing analysis, and Kelly sizing.
- Reproducible Python code repository with documented modules: `data.py`, `tree.py`, `greeks.py`, `edge.py`, `kelly.py`, `simulation.py`.
- Final presentation slides covering the end-to-end pipeline, key results, and failure cases.

### Optional

- Interactive Streamlit dashboard: input AMD strike/expiry → model price → edge → Kelly recommendation with live parameter sliders.
- Implied volatility surface plot for the AMD May 2026 chain.
- Calibration report: how sensitive is f\* to the choice of σ input?

---

## 13. Timeline and Milestones

| Week | Milestone |
|------|-----------|
| Week 1 | Literature review complete; AMD chain data extracted; risk-free rate and realized volatility sourced |
| Week 2 | CRR American binomial tree implemented; all four AMD tickets priced; early exercise boundary identified |
| Week 3 | Greeks computed numerically; validation against AMD deck; mispricing edge computed for all tickets |
| Week 4 | Kelly Criterion framework implemented; f\*, f_half, f_quarter computed; IV vs. realized volatility analysis |
| Week 5 | Monte Carlo P&L simulation complete; sensitivity analysis; all visualizations generated |
| Week 6 | Final research report drafted (10–20 pages); code repository cleaned; presentation slides prepared |

---

## 14. Main Challenges and Risks

- **IV staleness:** AMD IV of ~64–65% is a snapshot. If market conditions shift significantly, model inputs will be stale. *Mitigation:* Document exact timestamp; run sensitivity analysis on σ.
- **Model edge vs. true edge:** The CRR model's edge estimate depends on model correctness. If IV is wrong, the Kelly fraction is unreliable. *Mitigation:* Report Kelly fractions across a range of σ inputs; default to half-Kelly.
- **Kelly overbetting:** Full Kelly is optimal in theory but dangerous when probability estimates are uncertain. *Mitigation:* Always report and recommend fractional Kelly variants; never advocate full Kelly for real trades.
- **American put boundary instability:** Near-expiry American puts can exhibit rapid exercise boundary shifts that destabilize Greek estimates. *Mitigation:* Use centered finite differences with carefully selected δ.
- **Market microstructure noise:** Observed premiums reflect bid-ask spreads not captured by the model. *Mitigation:* Use limit price midpoints; report error ranges rather than point estimates.
- **Educational scope:** This is an analytical project. All outputs are educational illustrations. No real trades will be executed.

---

## 15. Possible Extensions and Future Improvements

- Extend the pipeline to the full AMD option chain (all strikes, multiple expiries) to build a complete mispricing surface.
- Implement a trinomial tree or Longstaff-Schwartz least-squares Monte Carlo as an alternative American pricer and compare prices.
- Add a regime detector (normal vs. herding) to condition the Kelly fraction on market microstructure signals — directly mirroring Project Suggestion 6's Step 4.
- Backtest the Kelly strategy over multiple AMD earnings cycles to evaluate the strategy's historical performance.
- Extend to a second ticker to test whether the IV overpricing pattern generalizes or is AMD-specific.
- Calibrate IV from the market price using an implied volatility solver (bisection or Brent's method) and compare the calibrated IV to the observed Fidelity IV.

---

## 16. Assumptions

- All four AMD contracts are American-style options. Early exercise is modeled at every node.
- No dividends. AMD's dividend yield is excluded from all model runs. The standard (no-dividend) risk-neutral probability formula is used throughout.
- Constant implied volatility. The IV of ~64–65% is held constant across the life of the option in all model runs. Stochastic volatility models are out of scope.
- The risk-free rate is constant and equal to the U.S. 3-month T-bill yield at the time of data collection (~5.3%).
- Market prices are taken from the Fidelity limit price column and treated as the observable market premium. Bid-ask midpoint is used where a single price is needed.
- The Kelly Criterion is applied analytically, not to a live portfolio. All sizing recommendations are illustrative.

---

## 17. Questions and Areas Where Feedback Is Needed

### Methodological Questions

- **Greeks Uncertainty:** Currently, the only Greek data used by the binomial model is IV (Implied Volatility). A key 
difference in my approach is to use additional Greeks in the model to see if it improves 
market predictions.

- **American Option Restriction:** Since I will restricted the model to American option contracts, how well can I account for
the possibility  of an early exercise of a contract before the expiry in the model. In addition,
not placing actual trades to see how the model compares to real trades, will I be able to train
the model appropriately?

- **Kelly probability conversion:** The Kelly formula requires a win probability p and net odds b. For options, the payoff is continuous, not binary. What is the most defensible way to discretize the payoff into a p-and-b structure — should the break-even price be used as the win/loss threshold?
- **Step count:** Is N=100 steps sufficient for American option convergence with T ≈ 13 days? Should N be scaled with T to keep Δt consistent across different expiry windows?
- **Edge threshold:** Is there a minimum edge threshold (e.g., |edge| > 2%) below which the Kelly fraction should be forced to zero to account for model uncertainty and transaction costs?

### Scope Questions

- **Monte Carlo benchmark:** Should a Monte Carlo simulation of the American option price (e.g., Longstaff-Schwartz) be added as a third pricing method to cross-validate the binomial model, or is this out of scope for six weeks?
- **Greeks validation granularity:** The AMD deck reports Greeks across the full strike chain ($262.50–$287.50). Should the validation cover all strikes, or only the four trade ticket strikes?

### Presentation & Reporting

- **Visualization priority:** Which visualization is most important for the final presentation: the early exercise boundary plot, the Kelly fraction table, or the IV vs. realized volatility chart?
- **Report length:** The proposal specifies 10–20 pages. Should the report lean toward the theory-heavy end (more mathematical derivations) or the empirical end (more results tables and plots)?

---

> **Disclaimer:** All model outputs in this project are educational illustrations and do not constitute investment advice. No real trades will be executed. Model assumptions — American exercise only, no dividends, constant implied volatility — are explicitly stated and must be accounted for before any real-world application.
